# Task A — Introduction and Theoretical Explanation

In this assignment we'll be implementing a basic fully-connected feed-forward neural network using NumPy. We will then be using it to classify characters in the MNIST dataset.

By fully-connected we mean that the output for each unit in the prior layer is given as input for each unit in the following layer.

By feed-forward we mean that the model has a series of layers in which the output of one layer, either fully or partially, is used as input for a subsequent layer, but not any prior layer.

Below is a diagram of a very simple fully-connected feed-forward neural network. Each layer in the diagram is represented by a column of dots, which represent units.

<div style="text-align: center;">
  <img src="imgs/nn-diagram.png" alt="Fully-connected feed-forward neural network diagram" width="400"/>
  <p style="text-align: center;">
    Source: Gow, Stephen & Niranjan, Mahesan & Pearman-Kanza, Samantha & Frey, Jeremy. (2022).
    A Review of Reinforcement Learning in Chemistry. <em>Digital Discovery. 1.</em> 10.1039/D2DD00047D.
  </p>
</div>

Mathematically we can think of these networks as a series of composed functions, expressed as:

$$ o = \color{#00b446}{f_3(}\color{#4040b8}{f_2(}\color{purple}{f_1(}\color{#f95b00}{x}\color{purple}{)}\color{#4040b8}{)}\color{#00b446}{)} $$

where
- $x$ is the input
- $f_i$ is the $i^{th}$ fully-connected layer
- $o$ is the output of the network

Note that the above diagram has three layers in total, two hidden layers and an output layer. Each layer has multiple units which receive input and send their output to the following layer. Each unit is comprised of two components: a linear function and an activation function. In mathematical notation we could state this as:

$$ o = a(w_1x_1 + w_2x_2 + \ldots + w_nx_n + b) $$

where
- $o$ is the output of the unit
- $a$ is the activation function, a non-linear function that we discuss more later in the homework
- $w_i$ is the $i^{th}$ weight of the linear function
- $x_i$ is the $i^{th}$ component of the input
- $b$ is the bias term

or rather, expressing the output of the whole layer,

$$ o^{(i)} = a(W^{(i)}x^{(i)} + b^{(i)}) $$

where
- $o^{(i)}$ is the output of the layer, a one dimensional vector with the same length as the number of units in the layer
- $x^{(i)}$ is the input into the $i^{th}$ layer, which is often equal to the output of the previous layer $o^{(i-1)}$
- $W^{(i)}$ is a matrix with dimensions `(input_size, num_units)` that holds the weights of all linear units in the layer
- $b^{(i)}$ is the bias term which is a one dimensional vector with the same length as the number of units in the layer

Let's combine these formulations to express the above network in terms of **functional composition**. The components of a given layer are shown in the same color.

$$
o = \color{#00b446}a^{(3)}(W^{(3)}\color{#4040b8}a^{(2)}(W^{(2)}\color{purple}a^{(1)}(W^{(1)}(\color{#f95b00}x\color{purple}) + b^{(1)})\color{#4040b8} + b^{(2)})\color{#00b446} + b^{(3)})
$$

From this you should have an idea of how an input is evaluated by this kind of neural network. In this homework we start by writing classes for each of the network's components and then combining them at the end with a `Model` class. Each of the classes we'll be implementing today will have a `forward` method which will do the portion of the above computation for the respective component.

We will also implement a `backward` method which is used during training and will compute the gradient of `forward` as a part of a larger gradient computation.

To make a network useful and accurately classifying inputs we have to adjust and tune all of the weights and biases in each unit. To do this we'll use a method called stochastic gradient descent (SGD) with mini-batches. This technique uses a loss function, often denoted with $\mathcal{L}$, which measures how 'wrong' the network is. The goal of training is to adjust and tune the weights and biases of the network such that $\mathcal{L}$ is minimized. To this end we calculate the gradient of $\mathcal{L}$ with respect to some single weight or bias, which tells us which way we should adjust a parameter to reduce the value of $\mathcal{L}$. In mathematical notation, where $z$ is the output of a linear function, or rather the pre-activation value of the unit:

$$ \frac{\partial\mathcal{L}}{\partial w} = \frac{\partial\mathcal{L}}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w} \;\;\; \text{or} \;\;\; \frac{\partial\mathcal{L}}{\partial b} = \frac{\partial\mathcal{L}}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial b}$$

To calculate the gradient of the weights we'll utilize the chain rule of derivation:

$$ \frac{d}{dx}f(g(h(x))) = f'(g(h(x))) \cdot g'(h(x)) \cdot h'(x)$$

and when calculating the gradient of the activation function we'll use the additive rule of derivation as well:

$$ \frac{d}{dx}(f(x) + g(x)) = f'(x) + g'(x)$$

Note from the above rules that we will need to be able to calculate the first derivative of every component with respect to one of the inputs. Practically, our implementation will calculate the derivative for every input in parallel, or rather the gradient, which is what `backward` does.

Training a network is an iterative process where we find the network's current outputs on some labeled data samples (`forward`), find the gradients of each parameter (`backward`), and adjust each parameter so the value of the loss function decreases for said inputs (`step`), usually until the values of the loss function stabilize at a minimum.

## The `Module` base class

Every layer, loss, and model in this assignment inherits from a tiny `Module` base class, defined in `src/nn/module.py`. This class is provided — you do not need to modify it. It mirrors the calling conventions of PyTorch's `nn.Module`:

- `module(x)` is short for `module.forward(x)`.
- `module.parameters()` yields `(param, grad)` pairs that `step` consumes for SGD updates.
- Subclasses override `forward` and `backward`. Subclasses that hold trainable weights also override `parameters`.
- The default `step(lr)` walks `self.parameters()` and applies one in-place SGD step `param -= lr * grad`, so subclasses that expose their parameters get the SGD update for free.
- `Module.__init__` sets up the `training` flag (used by layers like `Dropout`), so **every subclass constructor you write must call `super().__init__()` as its very first line.**

In the tasks that follow you will inherit from `Module` and override the relevant methods.

There are no tasks to complete in this section. Read on to the next task to begin implementing.


# Task B — Linear class

The linear layer contains multiple linear units. Within a network a linear layer will have an input size equal to the output size of the previous layer, and an output size equal to the number of linear units within the layer.

Below you will implement the `__init__`, `forward`, `backward`, and `parameters` methods of the layer in `src/nn/linear.py`. Assume that the input shape to `forward` or `backward` will be of `(num_samples, in_features)`.

The `Linear` class inherits from the `Module` base class, so `Linear(x)` is short for `Linear.forward(x)`.

#### `__init__(self, in_features: int, out_features: int, seed: int | None = None) -> None:`

Call `super().__init__()` first, then initialize the following attributes:

The layer should contain the following attributes at minimum:

- `self.W` — A two dimensional numpy array of shape `(in_features, out_features)` that contains the weights of all linear units in the layer. The initial values of these weights should be initialized to random values from a Gaussian distribution with $\mu = 0$ and $\sigma = \sqrt{\frac{2}{\mathrm{in\_features}}}$. This initialization method is known as He initialization. Use `np.random.default_rng(seed)` to construct a generator so the initialization is reproducible.
- `self.b` — A one dimensional numpy array of shape `(out_features,)` that contains the biases for each linear unit, initialized to zero. It will broadcast across the batch dimension during the forward pass.
- _Gradient and cache attributes_: To increase readability we declare attributes that will be used to store intermediate data required for each method but are not directly set in `__init__`. Initialize the gradients `self.dW` and `self.db` to zero arrays of the right shape, and set `self.x = None`, which will hold the input from the most recent `forward` call.

#### `forward(self, x: np.ndarray) -> np.ndarray:`

- **Input:** Some ndarray of shape `(num_samples, in_features)`
- Save the passed input to `self.x`, as this is required when calculating the gradient in `backward`.
- **Returns:** The linear transformation of passed inputs.

The output of the $j^{\text{th}}$ linear unit for one input $x$ is defined as:

$$
z_j = w_{1j}x_1 + w_{2j}x_2 + \ldots + w_{nj}x_n + b_j
$$

Note that for an input of size `(num_samples, in_features)` a calculation of this kind will be done `num_samples * out_features` times for a given call to forward.

#### `backward(self, dout: np.ndarray) -> np.ndarray:`

- **Input:** gradient of the loss function with respect to every output value of its own units from the previous call to `forward`.
- Calculate and internally save the gradient of the loss function with respect to each weight and bias in the layer (`self.dW` and `self.db`).
- Raise `RuntimeError` if `forward` has not been called yet.
- **Returns:** the gradient of the loss function with respect to its own inputs.

Below are the gradient calculations for individual weight, bias, and input values, where $i$ is the weight index within a linear unit, $j$ is the index of the unit within the layer, and $k$ is the index of a sample within the input.

$$
\frac{\partial \mathcal{L}}{\partial w_{ij}} = \frac{\partial \mathcal{L}}{\partial z_j} \cdot \frac{\partial z_j}{\partial w_{ij}} = \frac{\partial \mathcal{L}}{\partial z_j} * x_i \;\; \text{or} \;\; \sum_k \frac{\partial \mathcal{L}}{\partial z_{jk}} * x_{ik}
$$

$$
\frac{\partial \mathcal{L}}{\partial b_j} = \frac{\partial \mathcal{L}}{\partial z_j} \cdot \frac{\partial z_j}{\partial b_j} = \frac{\partial \mathcal{L}}{\partial z_j} \;\; \text{or} \;\; \sum_k \frac{\partial \mathcal{L}}{\partial z_{jk}}
$$

$$
\frac{\partial \mathcal{L}}{\partial x_{ik}} = \sum_j \frac{\partial \mathcal{L}}{\partial z_{jk}} \cdot \frac{\partial z_{jk}}{\partial x_{ik}} = \sum_j \frac{\partial \mathcal{L}}{\partial z_{jk}} * w_{ij}
$$

All of this can be done via matrix multiplications.

#### `parameters(self) -> Iterable[tuple[np.ndarray, np.ndarray]]:`

Yield the layer's `(parameter, gradient)` pairs in the order `(self.W, self.dW)` then `(self.b, self.db)`. The optimizer (introduced in Task G) iterates over these pairs and applies one update step:

$$
\begin{align*}
w_{ij} \leftarrow w_{ij} - \eta \frac{\partial \mathcal{L}}{\partial w_{ij}}
&\;\;\;\text{and}\;\;\;&
b_{j} \leftarrow b_{j} - \eta \frac{\partial \mathcal{L}}{\partial b_{j}}
\end{align*}
$$

where $\eta$ is the learning rate.

#### Tips, Hints, and Sanity Checks

- Your gradient calculations during `backward` should be the same `shape` as the attribute it is in respect to (e.g. `self.dW.shape == self.W.shape`).
- Consider the desired `shape` of any attribute you are calculating. Considering the shapes of other attributes and how `@` and `+` operations transform shape, how could we create a value with the desired shape?
- When writing gradients in `backward`, accumulate in-place: either `self.dW += new_grads` or `self.dW[...] = self.dW + new_grads`. Both update the existing array, so any other variable holding a reference to `self.dW` — such as the optimizer — automatically sees the updated gradient. A bare `self.dW = new_grads` rebinds the Python variable to a new array and silently breaks that reference.

## Deliverables

- Implement the four methods in `src/nn/linear.py`.
- Run `make test-b` and confirm the smoke tests pass.
- Run `make submit_b` to generate `submission.json` and upload it on the course webpage.


# Task D — Loss functions

The loss function is a measure of how far off the results of the network are from the actual results. The particular function that is used is often dependent on the application and type of prediction being made. When we're trying to predict some real-valued number, we would often use average mean squared error. For categorical data, which is what is found in the MNIST dataset, we'll use a cross entropy loss.

## Categorical encoding

First though, we need to consider how we'll represent this categorical information. It's often the case when working with processed training data that different categories will be represented by a number, which can be thought of as the index of some list of categorical labels. This is what the MNIST dataset provides, however, given that the labels of the MNIST dataset are those of handwritten digits, we have the convenient and _coincidental_ benefit that the index for each category has a very literal relationship to the category label.

Categorical models will often output a discrete probability distribution over all categories, literally a list of numbers that sum to one and is as long as the number of categories, each number estimating the likelihood that the input is a member of a particular category rather than one of the others. Given this output from our model, it can be helpful to think of our labels as being in this format. Since we assume the labels of our training data to be correct with $p=1.0$ each label will be a list with a single `1` in the index of the correct category. This is often called **one-hot encoding**. For example, for the category at index 3, which in our case is _coincidentally_ the digit "3", we have:

`3 = [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`

The above list is of length 10 because we are classifying into 10 different categories.

To be clear, during implementation it is not necessary to directly convert each label into this format, but the output of the loss function should be the same as if they were in this format. Our `CrossEntropy` class will take integer class indices, matching PyTorch's `nn.CrossEntropyLoss`.

## Softmax

One problem we have is that the raw output of the network, called logits and denoted by $\hat y$, almost never sums to 1 and the values will likely not all be in $[0,1]$, as is a requirement for a probability distribution. We can, however, force this property by normalizing the output. To do this we'll apply a **softmax** function to $\hat y$. The $i^{th}$ entry the distribution $q$, which is $\hat y$ with softmax applied is:

$$
q_i = \frac{e^{\hat y_i}}{\sum_{j} e^{\hat y_j}}
= \frac{e^{-\max_{j} \hat y_j}}{e^{-\max_{j} \hat y_j}} \cdot \frac{e^{\hat y_i}}{\sum_{j} e^{\hat y_j}}
= \frac{e^{\hat y_i - \max_{j} \hat y_j}}{\sum_{k} e^{\hat y_k - \max_{j} \hat y_j}}
$$

For numerical stability we will multiply both the numerator and denominator by $e^{-\max_j \hat y_j}$. This is due to the potential for exponents of large numbers to overflow into `inf`, which would break the training.

Since $e^x$ is monotonically increasing, or rather only increases as $x$ increases, we still have the same relative ordering of entries in the output, although the relative difference between the entries has changed. This relative ordering is important, as during inference we can still find the most likely category by using the largest value in the output without applying the softmax.

## Cross-entropy

The loss function $\mathcal{L}(p, q)$ we'll be using is called **Cross-entropy Loss**. Cross-entropy $H(p, q)$ measures the difference between two probability distributions and is expressed as such:

$$
H(p, q) = -\sum^{K}_{k=1} p_k \log( q_k)
$$

where
- $p$ is the true label with $p_k$ being the $k^{th}$ element of $p$.
- $q$ is the network output with the softmax applied, with $q_k$ being the $k^{th}$ element of $q$.
- And $K$ is the number of elements in $p$ and $q$

And the loss is the average cross-entropy across $N$ samples:

$$
\mathcal{L}(P, Q) = -\frac{1}{N}\sum^{N}_{n=1}\sum^{K}_{k=1} p_{n,k} \log( q_{n,k})
$$

where
- $N$ is the number of samples being calculated in the loss function.
- $P$ is the true distribution for a list of samples.
- $Q$ is the network's predicted distributions for the samples corresponding to $P$.
- $p_{n,k}$ is the $k^{th}$ element of the $n^{th}$ sample label in $P$.
- $q_{n,k}$ is the $k^{th}$ element of the $n^{th}$ network output with softmax in $Q$.


## Calculating the gradients of the loss

For training we must calculate the gradient of $\mathcal{L}(p, q)$ with respect to the network's outputs, or in mathematical notation:

$$
\frac{\partial \mathcal{L}}{\partial \hat y_i} = \frac{\partial \mathcal{L}}{\partial q_i} \cdot \frac{\partial q_i}{\partial \hat y_i} + \sum_{i \neq j} \frac{\partial \mathcal{L}}{\partial q_i} \cdot \frac{\partial q_i}{\partial \hat y_j}
$$

The partial derivative of $\mathcal{L}$ with respect to a single softmax value $q_i$, and the partial derivative of the softmax entry $q_i$ with respect to the network output for that entry $\hat y_j$ are stated below:

$$
\begin{align*}
\frac{\partial \mathcal{L}}{\partial q_i} =
\begin{cases}
    -\frac{1}{N} \cdot \frac{1}{q_i} & \text{if } i = y \\
    0 & \text{if } i \neq y
\end{cases}
& \; \; \; \; \; &
\frac{\partial q_i}{\partial \hat y_j} =
\begin{cases}
    q_i(1-q_i) & \text{if } i = j \\
    -q_i q_j & \text{if } i \neq j
\end{cases}
\end{align*}
$$

Putting these together we'll consider the cases of the logit with the correct class ($i = y$) and the logits of the incorrect classes ($i \neq y$).

$$
\frac{\partial \mathcal{L}}{\partial \hat y_i} =
\begin{cases}
    -\frac{1}{N} \cdot \frac{1}{q_y} \cdot q_y(1 - q_y) = -\frac{1 - q_y}{N} = \frac{q_y - 1}{N} & \text{if } i = y \\
    -\frac{1}{N} \cdot \frac{1}{q_y} \cdot (-q_i q_y) = \frac{q_i}{N} & \text{if } i \neq y
\end{cases}
$$

Considering what we know about one hot encoding, we can generalize this to:

$$
\frac{\partial \mathcal{L}}{\partial \hat y_i} = \frac{1}{N} (q_i - p_i)
$$

## Mean squared error

Cross-entropy is built for classification, where the target is a category and the network outputs a probability distribution. Many problems instead ask the network to predict a real-valued number (or a vector of them) — for example the price of a house or the coordinates of a point. For these **regression** problems we use **Mean Squared Error (MSE)**.

MSE measures the average of the squared differences between the prediction $\hat y$ and the target $y$. For a single sample with $K$ output values:

$$
\mathcal{L}(\hat y, y) = \frac{1}{K} \sum^{K}_{k=1} (\hat y_k - y_k)^2
$$

and across $N$ samples we average over every element, so the denominator is the total number of elements $N \cdot K$:

$$
\mathcal{L}(\hat Y, Y) = \frac{1}{N K} \sum^{N}_{n=1} \sum^{K}_{k=1} (\hat y_{n,k} - y_{n,k})^2
$$

Squaring does two things: it makes every difference positive (so errors do not cancel out), and it penalizes large errors much more heavily than small ones.

The gradient is straightforward because there is no softmax in the way. Differentiating the loss with respect to a single prediction $\hat y_i$ gives:

$$
\frac{\partial \mathcal{L}}{\partial \hat y_i} = \frac{2}{NK} (\hat y_i - y_i)
$$

In words: the gradient points away from the target, and its magnitude is proportional to how far off the prediction is.

## Comparing the two loss functions

Both classes return a single scalar from `forward` and the gradient of that scalar with respect to the network's output from `backward`, so they plug into the same training loop. The differences are in what they expect and what they are used for:

- **Task type.** Cross-entropy is for **classification**: the target is a category. MSE is for **regression**: the target is a real-valued number.
- **Targets.** `CrossEntropy` takes integer class indices of shape `(B,)`. `MSE` takes a real-valued target array with the **same shape** as the prediction.
- **Output of the network.** Cross-entropy applies a **softmax** internally to turn raw logits into a probability distribution before measuring the loss. MSE applies no such transformation — it compares the raw predictions to the targets directly.
- **Why not swap them.** Using MSE on a classification problem is possible but works poorly: it treats the output values as independent numbers rather than competing probabilities, and the gradient signal when the network is confidently wrong is weak. Cross-entropy paired with softmax gives a stronger, well-behaved gradient for classification, which is why it is the standard choice (and the one we use for MNIST).

## Writing the `CrossEntropy` Class

The `CrossEntropy` class inherits from the `Module` base class and lives in `src/nn/loss.py`. It has no trainable parameters, so you do not need to implement `parameters` or `step`.

#### `__init__(self) -> None`

- Call `super().__init__()` first.
- Declare attributes which will be set in `forward` and used by `backward`. Allocate `self.probs = None` and `self.y = None`.

#### `forward(self, logits: np.ndarray, y: np.ndarray) -> float`

- **Input:**
    - `logits` — The raw network output of shape `(B, C)`, before softmax is applied.
    - `y` — A `(B,)` array of integer class labels in `[0, C)`.
- Save the predicted probabilities (`self.probs`) and labels (`self.y`) for `backward`.
- Be sure to apply softmax to the input using the max-subtraction trick described above.
- **Return:** Value of the loss function as a Python `float`.

#### `backward(self) -> np.ndarray`

Note that, unlike a layer's `backward(dout)` which receives an upstream gradient from the next component, the loss's `backward()` takes **no argument** — the loss is where backpropagation begins, so it computes and returns dL/dz directly from its own cached state.

- **Input:** Nothing.
- Raise `RuntimeError` if `forward` has not been called yet.
- You will probably have to make a copy of your stored predicted probabilities.
- **Return:** Partial derivative of the loss function with regard to every input into the loss, of shape `(B, C)`.

## Writing the `MSE` Class

The `MSE` class also inherits from `Module` and lives in `src/nn/loss.py`. Like `CrossEntropy` it has no trainable parameters, so you do not need to implement `parameters` or `step`.

#### `__init__(self) -> None`

- Call `super().__init__()` first.
- Declare an attribute that will be set in `forward` and used by `backward`. Allocate `self.diff = None`.

#### `forward(self, pred: np.ndarray, target: np.ndarray) -> float`

- **Input:**
    - `pred` — The network's predictions, of any shape.
    - `target` — The target values, the **same shape** as `pred`.
- Save what you need for `backward`. Caching the difference `pred - target` (`self.diff`) is enough.
- Do **not** apply a softmax; compare the raw predictions to the targets directly.
- **Return:** The mean squared error across all elements as a Python `float`.

#### `backward(self) -> np.ndarray`

- **Input:** Nothing.
- Raise `RuntimeError` if `forward` has not been called yet.
- **Return:** Partial derivative of the loss with respect to every prediction, the same shape as `pred`. Remember the factor of `2 / N`, where `N` is the total number of elements.

## Deliverables

- Implement the methods in `src/nn/loss.py`.
- Run `make test-d` and confirm the smoke tests pass for both the `CrossEntropy` and `MSE` classes.
- Run `make submit_d` to generate `submission.json` and upload it on the course webpage.
